# Nettoyage du catalogue Steam

Ce notebook part du fichier brut `data/raw/` et produit un fichier propre dans `data/processed/`. Chaque décision que je vais prendre sera justifiée et chiffrée dans le journal de nettoyage, à la fin de ce notebook.

## 1. Chargement du dataset brut

- Je recharge le fichier d'origine, que je ne modifie jamais à la main.
- Je garde le nombre de lignes de départ dans `nb_depart` pour pouvoir chiffrer tout ce que je retire ensuite.

In [15]:
import pandas as pd

df = pd.read_csv("../data/raw/steam_games.csv")
nb_depart = len(df)
nb_depart

40833

## 2. Décision 1 : ne garder que les jeux

- Je regarde d'abord la nature des produits de la colonne `types`.

In [16]:
df["types"].value_counts(dropna=False)

types
app       38021
bundle     2572
sub         238
NaN           2
Name: count, dtype: int64

- Je regarde ensuite, pour chaque type, combien de dates de sortie manquent.

In [17]:
df.groupby("types", dropna=False)["release_date"].apply(lambda s: s.isna().sum())

types
app        367
bundle    2572
sub        238
NaN          2
Name: release_date, dtype: int64

- Les `bundle` (2 572) et les `sub` (238) n'ont aucune date de sortie : ce sont des lots de jeux, pas des jeux. Ils fausseraient mes questions sur les prix et sur les sorties par mois. Les 2 valeurs manquantes de `types` ne sont pas des jeux non plus.
- Je ne garde donc que les lignes `app` : 2 812 lignes retirées, il en reste 38 021.

In [18]:
nb_avant = len(df)
df = df[df["types"] == "app"].copy()
print(nb_avant - len(df), "lignes retirées, il reste", len(df))

2812 lignes retirées, il reste 38021


## 3. Décision 2 : les jeux sans nom

- Je compte les jeux dont le nom est manquant : il y en a 14 au total.

In [19]:
df["name"].isna().sum()

np.int64(14)

- Une ligne sans nom ne peut pas être identifiée ni analysée, donc je la supprime : il reste 38 007 jeux.

In [20]:
nb_avant = len(df)
df = df.dropna(subset=["name"]).copy()
print(nb_avant - len(df), "lignes retirées, il reste", len(df))

14 lignes retirées, il reste 38007


## 4. Décision 3 : les doublons

- Je teste combien de lignes ont à la fois le même nom et la même date de sortie : il y en a 4, soit 2 paires.
- Un même nom seul ne suffit pas : 91 lignes partagent un nom avec une autre, mais ce sont surtout des jeux différents (par exemple *Alter Ego* en 2010 et en 1986).

In [21]:
df.duplicated(subset=["name", "release_date"], keep=False).sum()

np.int64(4)

- J'affiche les 4 lignes concernées pour vérifier qu'il s'agit bien de doublons.

In [28]:
brut = pd.read_csv("../data/raw/steam_games.csv")
brut = brut[(brut["types"] == "app") & brut["name"].notna()]
doublons = brut[brut.duplicated(subset=["name", "release_date"], keep=False)]
doublons.sort_values("name")[["name", "release_date", "original_price", "developer"]]

,name,release_date,original_price,developer
1587,Batman: Arkham City - Game of the Year Edition,"Sep 7, 2012",$19.99,"Rocksteady Studios,Feral Interactive (Mac)"
17442,Batman: Arkham City - Game of the Year Edition,"Sep 7, 2012",$19.99,"Rocksteady Studios,Feral Interactive (Mac)"
5410,Warhammer Quest,"Jan 7, 2015",$14.99,"Rodeo Games,Twistplay"
19829,Warhammer Quest,"Jan 7, 2015",NaN,"Twistplay,Rodeo Games"


- *Batman: Arkham City - Game of the Year Edition* et *Warhammer Quest* apparaissent chacun deux fois avec la même date : c'est le même jeu listé deux fois.
- Je supprime ces 2 doublons en gardant la première ligne : il reste 38 005 jeux.

In [22]:
nb_avant = len(df)
df = df.drop_duplicates(subset=["name", "release_date"], keep="first").copy()
print(nb_avant - len(df), "lignes retirées, il reste", len(df))

2 lignes retirées, il reste 38005


- Je vérifie qu'il ne reste plus aucun doublon nom + date : le résultat doit être 0.

In [25]:
df.duplicated(subset=["name", "release_date"], keep=False).sum()

np.int64(0)

## Journal de nettoyage

| Quoi | Pourquoi | Lignes |
|---|---|---|
| Lignes `bundle`, `sub` et `types` manquant supprimées | Lots de jeux ou pages qui ne sont pas des jeux, sans date de sortie | 2 812 |
| Lignes sans `name` supprimées | Jeu non identifiable | 14 |
| Doublons nom + date supprimés | Même jeu listé deux fois | 2 |

Après ces 3 étapes, il reste 38 005 lignes.